# 05 — Hoi2Vec (THAM KHẢO, không thuộc bài nộp)


**Ý tưởng.** Trans2Vec dùng frequent itemset (FI), Hui2Vec dùng high-utility itemset (HUI). Ở đây ta giữ nguyên khung đó và thay pattern bằng **high occupancy itemset (HOI)** của bài *Deng, Mining high occupancy itemsets, FGCS 102 (2020) 222–229*.

- Occupancy (Definition 2): $O(P) = \sum_{T \ni P} \frac{|P|}{|T|}$
- P là HOI nếu $O(P) \ge \xi$ (Definition 3, ξ là giá trị tuyệt đối)

| Tên | Mô tả |
|---|---|
| TRANS-IT | Doc2Vec học từ item (giống notebook 02, 03) |
| TRANS-HOI | Doc2Vec học từ các HOI của giao dịch |
| **Hoi2Vec-IND** | trung bình vector của TRANS-IT và TRANS-HOI |
| **Hoi2Vec-JOI** | Doc2Vec học từ item và HOI cùng lúc |

"Hoi2Vec" là tên tự đặt, chưa có bài báo nào công bố phương pháp này. Cấu hình giống hệt notebook 02/03: PV-DBOW, d = 128, 50 epoch, Linear SVM C = 1, 10 lần chạy, cùng cách chia train/test.

**Liên hệ với utility.** Nếu đặt $u(i, T) = 1/|T|$ thì $u(P) = O(P)$. Vậy occupancy là một trường hợp riêng của utility không phụ thuộc giá. Khi đó transaction utility của mọi giao dịch bằng 1, nên cận TWU đúng bằng support (Theorem 1 của bài: $O(P) \le Sup(P)$).

In [1]:
import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import spearmanr

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "notebooks"))
sys.path.insert(0, str(ROOT / "tham_khao"))

# Only read-only helpers from utils.py are used; log_results is never called here.
from utils import (DATA, PROCESSED, RESULTS_CSV, read_labeled, read_utility_db, mine_fi, elbow,
                   item_docs, pattern_docs, joint_docs, separate_pattern_tokens, doc2vec,
                   run_embedding_methods, svm_score, baselines, nice, score_table, N_RUNS)
from hep import mine_hoi, occupancy_bruteforce

food_split = json.load(open(PROCESSED / "food_split.json"))
fm_split = json.load(open(PROCESSED / "foodmart_split.json"))
print("N_RUNS =", N_RUNS)

N_RUNS = 10


## 1. Cài đặt HEP và kiểm tra

`tham_khao/hep.py` cài Algorithm 1 của bài báo:

- occupancy-list = danh sách tid; giao dịch sắp theo độ dài tăng dần, nên mỗi list cũng sắp theo độ dài;
- k-itemset sinh từ hai (k−1)-ứng viên chung tiền tố, occupancy-list lấy bằng phép giao (Theorem 2);
- chỉ giữ ứng viên có $UBO(P) \ge \xi$, với $UBO(P) = \max_x \sum_{i \ge x} n_i \cdot l_x / l_i$ (Definition 7, Theorem 4).

Kiểm tra 1: ví dụ Table 1 của bài báo.

In [2]:
paper_db = [list("acd"), list("abd"), list("bcde"), list("ad"), list("cde"), list("abcde")]
occ = occupancy_bruteforce(paper_db)
check = pd.DataFrame({
    "Itemset": ["{e}", "{c, e}", "{c}"],
    "Bài báo": [0.783, 1.567, 1.12],
    "Tính được": [occ[frozenset("e")], occ[frozenset("ce")], occ[frozenset("c")]],
})
display(nice(check, digits=3))
hoi_1 = mine_hoi(paper_db, 1)
print("ξ = 1: {c,e} là HOI:", frozenset("ce") in hoi_1, "| {e} là HOI:", frozenset("e") in hoi_1, "(bài báo: True | False)")

Itemset,Bài báo,Tính được
{e},0.783,0.783
"{c, e}",1.567,1.567
{c},1.120,1.117


ξ = 1: {c,e} là HOI: True | {e} là HOI: False (bài báo: True | False)


Kết quả khớp với bài báo. Ví dụ này cũng cho thấy occupancy **không có tính chất downward closure**: {c, e} là HOI nhưng {e} thì không.

Kiểm tra 2: so HEP với cách vét cạn mọi tập con trên 50 CSDL ngẫu nhiên nhỏ, mỗi CSDL thử 4 mức ξ.

In [3]:
rng = np.random.default_rng(1)
n_ok = 0
for trial in range(50):
    db = [list(rng.choice(12, size=rng.integers(1, 8), replace=False)) for _ in range(40)]
    bf = occupancy_bruteforce(db)
    for xi in [0.5, 1, 2, 3]:
        assert {p for p, v in bf.items() if v >= xi - 1e-9} == set(mine_hoi(db, xi))
        n_ok += 1
print(f"{n_ok}/200 trường hợp: HEP cho đúng tập HOI như vét cạn")

200/200 trường hợp: HEP cho đúng tập HOI như vét cạn


## 2. Chọn ngưỡng ξ

**Vì sao phải lấy ξ > 1.** Với mọi giao dịch T, bản thân tập item của T có $O(T) \ge |T|/|T| = 1$. Nếu ξ ≤ 1 thì **mọi giỏ hàng đều là một HOI**. Các pattern này gần như chỉ xuất hiện trong đúng một giao dịch, nên chỉ đóng vai trò như mã định danh của giao dịch. Khi ξ > 1 thì $Sup(P) \ge O(P) > 1$, tức pattern phải xuất hiện trong ít nhất 2 giao dịch. Điều kiện này giống FI ở notebook 02, nơi Foodmart chọn ngưỡng ít nhất 2 giao dịch.

Food được giới hạn **tối đa 2 item**, như HUI ở notebook 03. Giỏ Food dài tới 250 item, và một itemset nằm trong hai giỏ dài vẫn có UBO > 1, nên cận UBO không cắt tỉa được và số ứng viên bùng nổ. Thử nghiệm với tối đa 3 item chạy quá 150 giây mà chưa xong.

In [4]:
fm = read_utility_db(DATA / "utility" / "foodmart.txt")
fm_trans = [t[0] for t in fm]
y_fm = pd.read_csv(PROCESSED / "foodmart_labels.csv")["label"].to_numpy()

docs_food, y_food = read_labeled(DATA / "food" / "food.txt")
food_trans = [list(dict.fromkeys(int(i) for i in d)) for d in docs_food]


def grid_table(trans, grid, max_len=None):
    rows, found = [], {}
    for xi in grid:
        t0 = time.time()
        found[xi] = mine_hoi(trans, xi, max_len=max_len)
        lens = [len(p) for p in found[xi]]
        rows.append({"ξ": xi, "Số HOI": len(lens), "HOI 1 item": sum(l == 1 for l in lens),
                     "HOI ≥ 2 item": sum(l >= 2 for l in lens), "Thời gian (s)": time.time() - t0})
    return pd.DataFrame(rows), found


fm_grid = [6, 5, 4, 3, 2.5, 2, 1.5, 1.25, 1.1]
fm_tab, fm_hois = grid_table(fm_trans, fm_grid)
display(nice(fm_tab, digits=2, caption="Foodmart: số HOI theo ξ"))

food_grid = [4, 3, 2.5, 2, 1.75, 1.5, 1.25, 1.1]
food_tab, food_hois = grid_table(food_trans, food_grid, max_len=2)
display(nice(food_tab, digits=2, caption="Food: số HOI (tối đa 2 item) theo ξ"))

print("Quy tắc elbow của notebook 02/03 chọn: Foodmart ξ =", elbow(fm_grid, list(fm_tab["Số HOI"])),
      "| Food ξ =", elbow(food_grid, list(food_tab["Số HOI"])))

ξ,Số HOI,HOI 1 item,HOI ≥ 2 item,Thời gian (s)
6.00,5,5,0,1.49
5.00,24,24,0,1.92
4.00,123,123,0,2.25
3.00,513,513,0,2.38
2.50,826,826,0,2.46
2.00,"1,181","1,175",6,2.49
1.50,"1,493","1,419",74,2.51
1.25,"1,751","1,503",248,2.55
1.10,"1,892","1,523",369,2.53


ξ,Số HOI,HOI 1 item,HOI ≥ 2 item,Thời gian (s)
4.00,5,5,0,14.74
3.00,272,272,0,14.77
2.50,842,842,0,14.50
2.00,"1,388","1,388",0,14.99
1.75,"1,498","1,498",0,16.03
1.50,"1,541","1,541",0,16.10
1.25,"1,559","1,555",4,14.72
1.10,"1,569","1,558",11,15.23


Quy tắc elbow của notebook 02/03 chọn: Foodmart ξ = 5 | Food ξ = 3


Quy tắc elbow (ngưỡng đầu tiên mà số pattern tăng gấp đôi) **không dùng được** ở đây. Ở ξ cao, số HOI tăng từ 0 lên vài cái, tỉ lệ tăng rất lớn, nên quy tắc chọn ngay một ngưỡng chỉ có vài HOI. Đường cong không có "khuỷu" rõ ràng trong vùng ξ > 1.

Vì vậy ta chọn **ξ nhỏ nhất trong lưới mà vẫn > 1, tức ξ = 1,1**, cho cả hai dataset. Cách này giống notebook 02 trên Foodmart, nơi elbow cũng rơi vào mức thấp nhất (2 giao dịch).

Với Foodmart, ta chạy thêm **ξ = 2** để kiểm tra độ nhạy. Theo Corollary 1 của bài, mọi HOI ở ξ = 2 đều là FI ở ngưỡng 2 giao dịch, tức là một tập con của chính tập FI mà notebook 02 đã dùng.

**Nhận xét quan trọng:** ở mọi ξ > 1, HOI gần như chỉ gồm **item đơn**. Trên Food, ξ = 1,1 cho 1.558 HOI một item và chỉ 11 cặp. Nguyên nhân là pattern nhiều item rất hiếm trong cả hai dataset: support thấp, và Food còn có giỏ dài làm |P|/|T| nhỏ. Do đó HOI chủ yếu đóng vai trò **lọc item**, giữ lại các item "chiếm chỗ" nhiều trong giỏ ngắn.

In [5]:
XI = 1.1
fm_xi_list = [XI, 2]


def docs_for(trans, item_tokens, hois):
    pats = list(hois.keys())
    d_hoi = pattern_docs(trans, pats)
    return d_hoi, joint_docs(item_tokens, d_hoi)


summary = []
for name, trans, hois in [("Foodmart", fm_trans, fm_hois[XI]), ("Food", food_trans, food_hois[XI])]:
    d_hoi, _ = docs_for(trans, item_docs(trans), hois)
    summary.append({"Dataset": name, "ξ": XI, "Số HOI": len(hois),
                    "HOI ≥ 2 item": sum(len(p) >= 2 for p in hois),
                    "Giao dịch không chứa HOI nào": sum(len(d) == 0 for d in d_hoi)})
display(nice(pd.DataFrame(summary)))

Dataset,ξ,Số HOI,HOI ≥ 2 item,Giao dịch không chứa HOI nào
Foodmart,1.10,"1,892",369,0
Food,1.10,"1,569",11,0


## 3. Foodmart — Hoi2Vec

In [6]:
fm_docs_it = item_docs(fm_trans)
tr_fm, te_fm = np.array(fm_split["train"]), np.array(fm_split["test"])
fm_res = {}
for xi in fm_xi_list:
    d_hoi, d_joi = docs_for(fm_trans, fm_docs_it, fm_hois[xi])
    fm_res[xi] = run_embedding_methods(fm_docs_it, d_hoi, d_joi, y_fm, tr_fm, te_fm,
                                       pattern_name="HOI", family="Hoi2Vec")
    display(score_table(fm_res[xi], caption=f"Foodmart, ξ = {xi}"))

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


10 lần chạy, 79 giây


Phương pháp,Accuracy (%),F1-macro (%),Số lần chạy
TRANS-IT,86.39 ± 0.38,86.38 ± 0.38,10
TRANS-HOI,85.16 ± 0.21,85.14 ± 0.22,10
Hoi2Vec-IND,85.58 ± 0.25,85.56 ± 0.26,10
Hoi2Vec-JOI,82.77 ± 0.32,82.52 ± 0.33,10


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


10 lần chạy, 64 giây


Phương pháp,Accuracy (%),F1-macro (%),Số lần chạy
TRANS-IT,86.38 ± 0.27,86.37 ± 0.27,10
TRANS-HOI,81.86 ± 0.17,81.85 ± 0.17,10
Hoi2Vec-IND,84.29 ± 0.44,84.26 ± 0.44,10
Hoi2Vec-JOI,80.94 ± 0.41,80.57 ± 0.43,10


## 4. Food — Hoi2Vec

In [7]:
tr_food, te_food = np.array(food_split["train"]), np.array(food_split["test"])
food_d_hoi, food_d_joi = docs_for(food_trans, docs_food, food_hois[XI])   # same item tokens as notebook 02
food_res = run_embedding_methods(docs_food, food_d_hoi, food_d_joi, y_food, tr_food, te_food,
                                 pattern_name="HOI", family="Hoi2Vec")
display(score_table(food_res, caption=f"Food, ξ = {XI} (HOI tối đa 2 item)"))

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_fl

10 lần chạy, 95 giây


Phương pháp,Accuracy (%),F1-macro (%),Số lần chạy
TRANS-IT,66.85 ± 0.81,66.78 ± 0.81,10
TRANS-HOI,66.64 ± 0.84,66.56 ± 0.84,10
Hoi2Vec-IND,66.91 ± 0.47,66.84 ± 0.47,10
Hoi2Vec-JOI,68.25 ± 0.90,68.25 ± 0.91,10


## 5. So sánh với FI và HUI (đọc `results.csv`, không ghi)

Số liệu FI và HUI lấy từ notebook 02 và 03. Bảng dưới cho thấy accuracy (%) trung bình 10 lần chạy.

In [8]:
res = pd.read_csv(RESULTS_CSV)


def pick(ds, setting, method):
    r = res[(res["dataset"] == ds) & (res["setting"] == setting) & (res["method"] == method)]
    return f"{r['AC'].iloc[0]:.2f} ± {r['AC_std'].iloc[0]:.2f}" if len(r) else "—"


def mine(df, method):
    r = df[df["method"] == method]
    return f"{r['AC'].iloc[0]:.2f} ± {r['AC_std'].iloc[0]:.2f}"


rows = []
for ds, hoi in [("Food", food_res), ("Foodmart", fm_res[XI])]:
    rows += [
        {"Dataset": ds, "Pattern": "— (chỉ item)", "TRANS-<P>": pick(ds, "Trans2Vec", "TRANS-IT"), "IND": "—", "JOI": "—"},
        {"Dataset": ds, "Pattern": "FI (Trans2Vec)", "TRANS-<P>": pick(ds, "Trans2Vec", "TRANS-FI"),
         "IND": pick(ds, "Trans2Vec", "Trans2Vec-IND"), "JOI": pick(ds, "Trans2Vec", "Trans2Vec-JOI")},
        {"Dataset": ds, "Pattern": "HUI (Hui2Vec)", "TRANS-<P>": pick(ds, "Hui2Vec", "TRANS-HUI"),
         "IND": pick(ds, "Hui2Vec", "Hui2Vec-IND"), "JOI": pick(ds, "Hui2Vec", "Hui2Vec-JOI")},
        {"Dataset": ds, "Pattern": f"HOI (Hoi2Vec, ξ = {XI})", "TRANS-<P>": mine(hoi, "TRANS-HOI"),
         "IND": mine(hoi, "Hoi2Vec-IND"), "JOI": mine(hoi, "Hoi2Vec-JOI")},
        {"Dataset": ds, "Pattern": "LENGTH-ONLY", "TRANS-<P>": pick(ds, "Contribution", "LENGTH-ONLY"), "IND": "—", "JOI": "—"},
    ]
rows.insert(9, {"Dataset": "Foodmart", "Pattern": "HOI (Hoi2Vec, ξ = 2)", "TRANS-<P>": mine(fm_res[2], "TRANS-HOI"),
                "IND": mine(fm_res[2], "Hoi2Vec-IND"), "JOI": mine(fm_res[2], "Hoi2Vec-JOI")})
display(nice(pd.DataFrame(rows), caption="Accuracy (%) — FI, HUI, HOI"))

Dataset,Pattern,TRANS-,IND,JOI
Food,— (chỉ item),66.80 ± 0.93,—,—
Food,FI (Trans2Vec),69.75 ± 0.77,68.10 ± 0.77,70.80 ± 0.75
Food,HUI (Hui2Vec),69.75 ± 0.79,67.66 ± 1.50,70.34 ± 0.65
Food,"HOI (Hoi2Vec, ξ = 1.1)",66.64 ± 0.84,66.91 ± 0.47,68.25 ± 0.90
Food,LENGTH-ONLY,73.38 ± 0.00,—,—
Foodmart,— (chỉ item),86.29 ± 0.36,—,—
Foodmart,FI (Trans2Vec),85.89 ± 0.27,86.21 ± 0.23,82.74 ± 0.46
Foodmart,HUI (Hui2Vec),78.26 ± 0.54,87.81 ± 0.42,82.58 ± 0.77
Foodmart,"HOI (Hoi2Vec, ξ = 1.1)",85.16 ± 0.21,85.58 ± 0.25,82.77 ± 0.32
Foodmart,"HOI (Hoi2Vec, ξ = 2)",81.86 ± 0.17,84.29 ± 0.44,80.94 ± 0.41


## 6. Occupancy và độ dài giỏ hàng

Notebook 04 cho thấy nhãn Food gần như do số lượt mua quyết định, mà số lượt mua thể hiện qua độ dài giỏ. Occupancy chia cho |T|, nên ta xem hai điều:

1. **HOI lấy từ giỏ nào, và mỗi giỏ có bao nhiêu token pattern?** Ta tính độ dài trung bình của giỏ chứa token (trung bình theo từng cặp token–giỏ) và hệ số Spearman giữa độ dài giỏ với số token pattern. Nếu tương quan rất cao, embedding pattern vẫn "nhìn thấy" độ dài.
2. **Trong cùng một mức độ dài, embedding có phân loại tốt hơn đoán theo lớp đa số không?**

In [9]:
docs_fi_food, _ = read_labeled(DATA / "food" / "food_trans_fis_0.002.txt")    # FIs used in notebook 02
fm_fi = mine_fi(fm_trans, 2 / len(fm_trans))                                    # FIs used in notebook 02
fm_docs_fi = pattern_docs(fm_trans, list(fm_fi.keys()))
fm_d_hoi, _ = docs_for(fm_trans, fm_docs_it, fm_hois[XI])

rows = []
for ds, trans, sets in [("Food", food_trans, [("FI (δ = 0,2%)", docs_fi_food), (f"HOI (ξ = {XI})", food_d_hoi)]),
                        ("Foodmart", fm_trans, [("FI (≥ 2 giao dịch)", fm_docs_fi), (f"HOI (ξ = {XI})", fm_d_hoi)])]:
    L = np.array([len(t) for t in trans])
    rows.append({"Dataset": ds, "Tập token": "toàn bộ giỏ", "Độ dài TB của giỏ chứa token": L.mean(),
                 "Số token TB mỗi giỏ": np.nan, "Spearman(độ dài, số token)": np.nan})
    for name, docs in sets:
        n_tok = np.array([len(d) for d in docs])
        rows.append({"Dataset": ds, "Tập token": name,
                     "Độ dài TB của giỏ chứa token": (L * n_tok).sum() / n_tok.sum(),
                     "Số token TB mỗi giỏ": n_tok.mean(),
                     "Spearman(độ dài, số token)": spearmanr(L, n_tok)[0]})
display(nice(pd.DataFrame(rows), digits=2))

Dataset,Tập token,Độ dài TB của giỏ chứa token,Số token TB mỗi giỏ,"Spearman(độ dài, số token)"
Food,toàn bộ giỏ,25.87,nan,nan
Food,"FI (δ = 0,2%)",78.29,43.65,0.99
Food,HOI (ξ = 1.1),61.97,25.89,1.00
Foodmart,toàn bộ giỏ,4.42,nan,nan
Foodmart,FI (≥ 2 giao dịch),5.45,4.50,0.99
Foodmart,HOI (ξ = 1.1),5.44,4.57,0.98


Tiếp theo, ta kiểm tra trên Food xem embedding có mang thông tin **ngoài độ dài** hay không. Test set được chia thành 4 nhóm theo tứ phân vị độ dài giỏ. Trong mỗi nhóm, độ dài gần như không đổi, nên:

- *Lớp đa số* cho biết accuracy đạt được nếu chỉ biết giỏ thuộc nhóm độ dài nào.
- Nếu embedding vượt rõ lớp đa số **bên trong từng nhóm**, pattern mang thông tin ngoài độ dài.

Mô hình giống notebook 02: Linear SVM C = 1, học trên toàn bộ tập train; embedding lấy trung bình 3 seed.

In [10]:
from sklearn.svm import LinearSVC


def predict(X):
    clf = LinearSVC(C=1.0, max_iter=5000).fit(X[tr_food], y_food[tr_food])
    return clf.predict(X[te_food])


len_food = np.array([len(d) for d in docs_food])
docs_fi_joi = joint_docs(docs_food, separate_pattern_tokens(docs_fi_food))     # same docs as notebook 02
SEEDS = [0, 1, 2]
preds = {"LENGTH-ONLY": [predict(np.log1p(len_food)[:, None].astype(float))]}
for name, docs in [("Trans2Vec-JOI (FI)", docs_fi_joi), ("Hoi2Vec-JOI (HOI)", food_d_joi)]:
    preds[name] = [predict(doc2vec(docs, seed=s)) for s in SEEDS]

y_te, L_te = y_food[te_food], len_food[te_food]
bins = pd.qcut(L_te, 4, duplicates="drop")
rows = []
for b in list(bins.categories) + ["Toàn bộ test"]:
    m = np.ones(len(L_te), bool) if b == "Toàn bộ test" else np.asarray(bins == b)
    _, cnt = np.unique(y_te[m], return_counts=True)
    row = {"Nhóm độ dài": str(b), "Số giỏ": int(m.sum()), "% yes": 100 * (y_te[m] == "yes").mean(),
           "Lớp đa số (%)": 100 * cnt.max() / m.sum()}
    for name, ps in preds.items():
        row[name] = np.mean([100 * (p[m] == y_te[m]).mean() for p in ps])
    rows.append(row)
display(nice(pd.DataFrame(rows), digits=1, caption="Food: accuracy (%) trong từng nhóm độ dài giỏ (test set)"))

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Nhóm độ dài,Số giỏ,% yes,Lớp đa số (%),LENGTH-ONLY,Trans2Vec-JOI (FI),Hoi2Vec-JOI (HOI)
"(2.999, 9.0]",202,16.3,83.7,83.7,83.2,82.2
"(9.0, 17.0]",207,39.6,60.4,65.2,62.0,57.3
"(17.0, 31.0]",193,54.4,54.4,54.4,51.5,49.9
"(31.0, 228.0]",198,89.9,89.9,89.9,84.8,83.7
Toàn bộ test,800,49.8,50.2,73.4,70.5,68.3
